# Notebook 03: SageMaker Pipeline, Deployment, Verification and Baseline Comparison

Module: ITI113 Machine Learning & Operations 

Focus Area: C, MLOps (Pipeline, experiment tracking, model registry, deployment, verification)

---

## What this notebook does

1. Writes preprocess.py, train.py, and inference.py to a local src folder.
2. Defines and runs a SageMaker Pipeline: Process, Train, Condition, Register.
3. Keeps the SageMaker training container free of MLflow credentials and MLflow dependencies.
4. After a successful pipeline run, reads SageMaker job metadata and metrics, then logs them to the team's SageMaker Serverless MLflow App, validating the app TeamId tag first.
5. Registers a quality-approved model in SageMaker Model Registry, approves it, and deploys it as the champion serverless endpoint.
6. Verifies the deployed endpoint (InService check, Model Registry traceability, single and batch invocation).
7. Reruns the same pipeline with baseline (untuned) hyperparameters and deploys a second serverless endpoint, so the demo UI (Streamlit, via Notebook 06's relay) can show champion vs baseline live inference side by side.


---


In [31]:
%%capture
%pip install --upgrade "sagemaker>=2,<3" boto3 botocore mlflow sagemaker-mlflow
print("Packages installed.")

## 0. Configuration

Sets up the SageMaker session, S3 bucket and prefix, resolves the team's MLflow App config, and defines the pipeline names and quality gate used throughout this notebook.


In [32]:
import os
os.environ["SAGEMAKER_SUPPRESS_V2_WARNING"] = "1"

import warnings
warnings.filterwarnings("ignore")

import logging
logging.getLogger("sagemaker").setLevel(logging.ERROR)

import boto3
import sagemaker
import json
import time
from pathlib import Path

# AWS and SageMaker setup
session = sagemaker.Session()
role = sagemaker.get_execution_role()
region = boto3.Session().region_name

BUCKET = "nyp-26s1-iti113"
TEAM_ID = "team03"
STUDENT_ID = "s301"
COURSE = "ITI113"
SEMESTER = "26S1"
PROJECT_NAME = "crypto-scam-detector"
PREFIX = f"iti113/{TEAM_ID}/data/{PROJECT_NAME}"
PROCESSING_INSTANCE_TYPE = "ml.m5.large"
TRAINING_INSTANCE_TYPE = "ml.m5.large"

# SageMaker Serverless MLflow App setup
TEAM_CONFIG_FILE = Path(f"mlflow_app_config_{TEAM_ID}.json")
STUDENT_CONFIG_FILE = Path(f"mlflow_app_config_{TEAM_ID}_{STUDENT_ID}.json")

config_candidates = [
    TEAM_CONFIG_FILE,
    STUDENT_CONFIG_FILE,
    *sorted(Path(".").glob(f"mlflow_app_config_{TEAM_ID}_*.json")),
]

MLFLOW_APP_ARN = None
MLFLOW_EXPERIMENT_NAME = f"{COURSE}/{TEAM_ID}/Experiment1"
mlflow_config = {}
config_used = None

for config_file in config_candidates:
    if config_file.exists():
        mlflow_config = json.loads(config_file.read_text(encoding="utf-8"))
        MLFLOW_APP_ARN = (
            mlflow_config.get("MLFLOW_APP_ARN")
            or mlflow_config.get("mlflow_app_arn")
            or mlflow_config.get("arn")
        )
        MLFLOW_EXPERIMENT_NAME = (
            mlflow_config.get("EXPERIMENT_NAME")
            or mlflow_config.get("experiment_name")
            or MLFLOW_EXPERIMENT_NAME
        )
        config_used = config_file
        break

# Fallback only: team03's MLflow App ARN from setup.
DEFAULT_MLFLOW_APP_ARN = (
    "arn:aws:sagemaker:ap-southeast-1:044528205969:"
    "mlflow-app/app-J5AYUG4AJHVW"
)

if MLFLOW_APP_ARN is None:
    MLFLOW_APP_ARN = DEFAULT_MLFLOW_APP_ARN
    print(
        "[WARNING] No local MLflow config file found. "
        "Using DEFAULT_MLFLOW_APP_ARN. Make sure this ARN belongs to your own team."
    )
else:
    print(f"Loaded MLflow App config from {config_used}")

config_team_id = mlflow_config.get("TEAM_ID") or mlflow_config.get("team_id")
if config_team_id and config_team_id != TEAM_ID:
    raise ValueError(
        f"Config file team mismatch: config TEAM_ID={config_team_id}, notebook TEAM_ID={TEAM_ID}. "
        "Do not use another team's MLflow config."
    )

# Safety check for team-level MLflow restriction
sm_for_mlflow = boto3.client("sagemaker", region_name=region)

try:
    tag_response = sm_for_mlflow.list_tags(ResourceArn=MLFLOW_APP_ARN)
    mlflow_app_tags = {t["Key"]: t["Value"] for t in tag_response.get("Tags", [])}

    print("MLflow App tags:")
    for k, v in mlflow_app_tags.items():
        print(f"  {k}: {v}")

    app_team_id = mlflow_app_tags.get("TeamId")
    if app_team_id != TEAM_ID:
        raise PermissionError(
            f"MLflow App TeamId tag mismatch. App TeamId={app_team_id}, notebook TEAM_ID={TEAM_ID}. "
            "Do not log to another team's MLflow App."
        )

    print(f"[OK] MLflow App tag TeamId={app_team_id} matches notebook TEAM_ID={TEAM_ID}")

except Exception as e:
    print("\n[ERROR] Could not validate MLflow App team tag.")
    print("This usually means one of the following:")
    print("1. The MLflow App ARN belongs to another team and IAM correctly blocked access.")
    print("2. The MLflow App is missing the TeamId tag.")
    print("3. The current role lacks permission to list tags for this MLflow App.")
    print(type(e).__name__, e)
    raise

os.environ["MLFLOW_TRACKING_URI"] = MLFLOW_APP_ARN
os.environ["MLFLOW_EXPERIMENT_NAME"] = MLFLOW_EXPERIMENT_NAME


def create_mlflow_app_presigned_url(fragment: str = "") -> str:
    sm_for_mlflow = boto3.client("sagemaker", region_name=region)
    response = sm_for_mlflow.create_presigned_mlflow_app_url(
        Arn=MLFLOW_APP_ARN
    )

    base_url = response.get("AuthorizedUrl") or response.get("Url")

    if not base_url:
        raise RuntimeError(
            "create_presigned_mlflow_app_url did not return AuthorizedUrl or Url. "
            f"Response: {response}"
        )

    base_url = base_url.split("#", 1)[0]

    if fragment:
        return base_url + "#" + fragment.lstrip("#")

    return base_url


def print_mlflow_presigned_links(experiment_id=None, run_id=None):
    if experiment_id is not None:
        experiment_url = create_mlflow_app_presigned_url(
            f"/experiments/{experiment_id}"
        )
        print("Presigned MLflow experiment URL:")
        print(experiment_url)

    if experiment_id is not None and run_id is not None:
        run_url = create_mlflow_app_presigned_url(
            f"/experiments/{experiment_id}/runs/{run_id}"
        )
        print("\nPresigned MLflow run URL:")
        print(run_url)

PIPELINE_NAME = f"iti113-{TEAM_ID}-crypto-scam-detector"
MODEL_PACKAGE_GROUP = f"{TEAM_ID}-CryptoScamDetector"
ENDPOINT_NAME = f"iti113-{TEAM_ID}-crypto-scam-detector"
QUALITY_GATE_AUC = 0.85

s3_raw = boto3.client("s3", region_name=region)
try:
    pointer_obj = s3_raw.get_object(Bucket=BUCKET, Key=f"{PREFIX}/raw/_latest.json")
    raw_pointer = json.loads(pointer_obj["Body"].read())
    RAW_DATA_URI = f"s3://{BUCKET}/{raw_pointer['s3_key']}"
except s3_raw.exceptions.NoSuchKey:
    raise FileNotFoundError(
        f"No raw dataset version pointer found at s3://{BUCKET}/{PREFIX}/raw/_latest.json. "
        "Run Notebook 01 first to upload and version the raw dataset."
    )
PIPELINE_ROOT = f"s3://{BUCKET}/{PREFIX}/pipeline"

SCRIPTS_S3_PREFIX = f"{PREFIX}/pipeline_src"
SCRIPTS_S3_URI = f"s3://{BUCKET}/{SCRIPTS_S3_PREFIX}"
LOCAL_PIPELINE_SRC = "pipeline_src"

print(
    f"Pipeline: {PIPELINE_NAME}\n"
    f"Bucket: {BUCKET}\n"
    f"Team prefix: {PREFIX}\n"
    f"Semester: {SEMESTER}\n"
    f"Region: {region}\n"
    f"SageMaker role: {role}\n"
    f"MLflow App ARN: {MLFLOW_APP_ARN}\n"
    f"MLflow experiment: {MLFLOW_EXPERIMENT_NAME}\n"
    f"Raw data URI: {RAW_DATA_URI}\n"
    f"Pipeline source S3 URI: {SCRIPTS_S3_URI}\n"
    f"Local pipeline source: {LOCAL_PIPELINE_SRC}"
)

Loaded MLflow App config from mlflow_app_config_team03_s301.json
MLflow App tags:
  Semester: 26S1
  sagemaker:domain-arn: arn:aws:sagemaker:ap-southeast-1:044528205969:domain/d-gpdrdk2w4dgw
  ProjectName: crypto-scam-detector
  sagemaker:space-arn: arn:aws:sagemaker:ap-southeast-1:044528205969:space/d-gpdrdk2w4dgw/team03-shared
  Course: ITI113
  TeamId: team03
  CreatedByNotebook: 01A_setup_sagemaker_mlflow_app
  StudentId: s301
[OK] MLflow App tag TeamId=team03 matches notebook TEAM_ID=team03
Pipeline: iti113-team03-crypto-scam-detector
Bucket: nyp-26s1-iti113
Team prefix: iti113/team03/data/crypto-scam-detector
Semester: 26S1
Region: ap-southeast-1
SageMaker role: arn:aws:iam::044528205969:role/SageMakerExecutionRole-ITI113-Team03
MLflow App ARN: arn:aws:sagemaker:ap-southeast-1:044528205969:mlflow-app/app-J5AYUG4AJHVW
MLflow experiment: ITI113/team03/Experiment1
Raw data URI: s3://nyp-26s1-iti113/iti113/team03/data/crypto-scam-detector/raw/crypto_scam_dataset_v4.csv
Pipeline sourc

## 0A. Precheck SageMaker MLflow App Connection

Tests that this notebook can create or use the team's MLflow experiment before launching the pipeline, and checks the MLflow App has the correct TeamId tag. If this fails, resolve the MLflow App ARN, the sagemaker-mlflow package, or IAM permissions before continuing.


In [33]:
import mlflow
import time

mlflow.set_tracking_uri(MLFLOW_APP_ARN)
experiment = mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

with mlflow.start_run(run_name=f"{TEAM_ID}_pipeline_notebook_precheck_{int(time.time())}") as run:
    mlflow.set_tags({
        "course": "ITI113",
        "semester": SEMESTER,
        "team_id": TEAM_ID,
        "student_id": STUDENT_ID,
        "dataset": "crypto-scam-detector",
        "run_type": "sagemaker_pipeline_precheck",
        "tracking_backend": "sagemaker_mlflow_app",
        "mlflow_app_arn": MLFLOW_APP_ARN,
    })
    mlflow.log_param("source", "notebook_03_precheck")
    mlflow.log_metric("connection_success", 1)

    precheck_run_id = run.info.run_id
    precheck_experiment_id = run.info.experiment_id

print("SageMaker MLflow App precheck completed.")
print("MLflow App ARN:", MLFLOW_APP_ARN)
print("Experiment:", MLFLOW_EXPERIMENT_NAME)
print("Experiment ID:", precheck_experiment_id)
print("Run ID:", precheck_run_id)

🏃 View run team03_pipeline_notebook_precheck_1787128691 at: https://mlflow.sagemaker.ap-southeast-1.app.aws/#/experiments/1/runs/4bbc56400c954e768748dc91bf266c17
🧪 View experiment at: https://mlflow.sagemaker.ap-southeast-1.app.aws/#/experiments/1
SageMaker MLflow App precheck completed.
MLflow App ARN: arn:aws:sagemaker:ap-southeast-1:044528205969:mlflow-app/app-J5AYUG4AJHVW
Experiment: ITI113/team03/Experiment1
Experiment ID: 1
Run ID: 4bbc56400c954e768748dc91bf266c17


## 0B. Load Best Model Configuration

Reads best_model.json and queries the SageMaker MLflow App for that run's logged hyperparameters, so the pipeline trains whichever model type won the earlier comparison, Logistic Regression or Random Forest.


In [34]:
from mlflow.tracking import MlflowClient

BEST_MODEL_JSON = Path("best_model.json")
if not BEST_MODEL_JSON.exists():
    raise FileNotFoundError(
        "best_model.json was not found. Run Notebook 02's model-selection step first."
    )

best_model_info = json.loads(BEST_MODEL_JSON.read_text(encoding="utf-8"))
BEST_RUN_ID = best_model_info["best_run_id"]
BEST_RUN_NAME = best_model_info["best_run_name"]
BEST_AUC = best_model_info["best_auc"]

mlflow.set_tracking_uri(MLFLOW_APP_ARN)
mlflow_client = MlflowClient()
best_run_params = mlflow_client.get_run(BEST_RUN_ID).data.params

if "C" in best_run_params:
    MODEL_TYPE = "logistic_regression"
    MODEL_HYPERPARAMS = {
        "C": float(best_run_params["C"]),
        "penalty": best_run_params.get("penalty", "l2"),
        "solver": best_run_params.get("solver", "lbfgs"),
        "max_iter": int(float(best_run_params.get("max_iter", 1000))),
    }
elif "n_estimators" in best_run_params:
    MODEL_TYPE = "random_forest"
    MODEL_HYPERPARAMS = {
        "n_estimators": int(float(best_run_params["n_estimators"])),
        "max_depth": int(float(best_run_params["max_depth"])),
        "min_samples_leaf": int(float(best_run_params["min_samples_leaf"])),
    }
else:
    raise ValueError(
        f"Could not infer model type from run {BEST_RUN_NAME}'s logged params: {sorted(best_run_params)}"
    )

print(
    f"Best run: {BEST_RUN_NAME} (AUC={BEST_AUC:.4f})\n"
    f"Model type: {MODEL_TYPE}\n"
    f"Hyperparameters: {MODEL_HYPERPARAMS}"
)

Best run: lr_candidate_04 (AUC=0.8908)
Model type: logistic_regression
Hyperparameters: {'C': 10.0, 'penalty': 'l2', 'solver': 'lbfgs', 'max_iter': 1000}


## 1. Write Pipeline Scripts

SageMaker Pipeline steps run as isolated jobs, each in its own managed container.

The deployment bundle saved with joblib contains the trained model, the fitted TF-IDF vectorizer, the engineered indicator feature column names, and the full feature column order. This lets the deployed endpoint accept raw message text as JSON input and perform text cleaning, feature engineering and TF-IDF transformation inside inference.py before prediction.

The Processing step still handles training-time preprocessing, but no separate Processing job runs for live inference, the endpoint preprocesses in memory. MLflow logging happens outside the training container, from this notebook, after a successful pipeline run.


In [35]:
os.makedirs('src', exist_ok=True)
print('src/ directory ready')

src/ directory ready


In [36]:
%%writefile src/preprocess.py
"""SageMaker Processing Job for text preprocessing.

Cleans the raw scam-message text, builds engineered indicator features
(urgency, contact/link, structural characteristics), fits TF-IDF on the
training split only, and saves what training and serving need to stay in
sync: TF-IDF features (.npz), engineered features and labels (CSV), and a
preprocessor bundle (joblib) with the fitted vectorizer.
"""
import os
import re
import argparse
import glob

import pandas as pd
import numpy as np
import joblib
import scipy.sparse as sp

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

import subprocess
import sys

try:
    from rapidfuzz import fuzz
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "rapidfuzz"])
    from rapidfuzz import fuzz

parser = argparse.ArgumentParser()
parser.add_argument('--test-size',    type=float, default=0.20)
parser.add_argument('--random-state', type=int,   default=42)
args = parser.parse_args()

# Discovers the input CSV by filename pattern rather than hardcoding the dataset name
input_dir = '/opt/ml/processing/input'
csv_candidates = sorted(glob.glob(os.path.join(input_dir, '*.csv')))
if not csv_candidates:
    raise FileNotFoundError(
        f'No CSV file found in {input_dir}. Expected the file that triggered '
        'this pipeline run (or crypto_scam_dataset.csv for a manual run).'
    )
input_path = csv_candidates[0]
print(f'Using input file: {input_path}')
output_dir = '/opt/ml/processing/output'
os.makedirs(output_dir, exist_ok=True)

SCAM_LABEL = "scam"

# Text cleaning, mirrors utils/preprocessing.py's clean_text()
URL_PATTERN = re.compile(r"https?://[^\s]+|www\.[^\s]+")

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.strip()
    text = URL_PATTERN.sub(" <url> ", text)
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()

# Engineered indicator features, keyword lists follow the project proposal's indicator design
URGENT_KEYWORDS = [
    "urgent", "immediately", "hurry", "act now", "act fast", "limited time",
    "limited slots", "limited spots", "today only", "last chance",
    "final notice", "final warning", "don't miss", "don't wait",
    "expires", "expiring", "closing soon", "ending soon", "before it's too late",
    "respond now", "reply now", "confirm now", "verify now", "claim now",
    "while supplies last", "only a few left", "act before", "time-sensitive",
    "this offer", "exclusive offer", "one time offer",
]
GUARANTEED_RETURN_KEYWORDS = [
    "guaranteed", "guarantee", "risk-free", "risk free", "no risk",
    "zero risk", "sure profit", "sure win", "can't lose", "cannot lose",
    "double your money", "triple your money", "multiply your", "10x", "100x",
    "passive income", "easy money", "get rich", "financial freedom",
    "life-changing", "once in a lifetime", "secret method", "proven strategy",
    "insider information", "insider tip", "exclusive access", "vip access",
]
PAYMENT_KEYWORDS = [
    "deposit", "transfer", "send funds", "send payment", "send money",
    "top up", "top-up", "processing fee", "activation fee", "unlock fee",
    "advance fee", "small fee", "gas fee", "network fee", "wallet",
    "crypto", "bitcoin", "ethereum", "usdt", "stablecoin",
]
OFF_PLATFORM_KEYWORDS = [
    "telegram", "whatsapp", "discord", "signal", "wechat", "line app",
    "private chat", "private message", "dm me", "direct message",
    "text me", "call me", "contact me directly", "reach out privately",
    "add me on",
]
CREDENTIAL_KEYWORDS = [
    "seed phrase", "private key", "recovery phrase", "recovery key",
    "wallet password", "login credentials", "verification code", "otp",
    "security code", "pin number", "social security"
]

WALLET_PATTERN = re.compile(r"\b(?:0x[a-fA-F0-9]{40}|[13][a-km-zA-HJ-NP-Z1-9]{25,34})\b")
EMAIL_PATTERN = re.compile(r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}")
PHONE_PATTERN = re.compile(r"(?:\+?\d[\d\s-]{7,}\d)")
COUNTDOWN_PATTERN = re.compile(r"\b\d+\s*(?:hour|hours|hr|hrs|minute|minutes|min|mins|day|days)\b", re.IGNORECASE)

def find_keyword_matches_fuzzy(message, keywords, threshold=85):
    message_lower = message.lower()
    tokens = message_lower.split()
    matches = []
    for kw in keywords:
        kw_lower = kw.lower()
        if " " in kw_lower:
            if fuzz.partial_ratio(kw_lower, message_lower) >= threshold:
                matches.append(kw)
        else:
            if any(fuzz.ratio(kw_lower, tok) >= threshold for tok in tokens):
                matches.append(kw)
    return matches

def extract_engineered_features(text):
    if not isinstance(text, str):
        text = ""

    urgency_keyword_hits = len(find_keyword_matches_fuzzy(text, URGENT_KEYWORDS))
    guaranteed_return_hits = len(find_keyword_matches_fuzzy(text, GUARANTEED_RETURN_KEYWORDS))
    countdown_hits = len(COUNTDOWN_PATTERN.findall(text))
    exclamation_count = text.count("!")
    urgency_score = urgency_keyword_hits + countdown_hits + min(exclamation_count, 3)

    wallet_matches = WALLET_PATTERN.findall(text)
    url_matches = URL_PATTERN.findall(text)
    email_matches = EMAIL_PATTERN.findall(text)
    phone_matches = PHONE_PATTERN.findall(text)
    payment_hits = len(find_keyword_matches_fuzzy(text, PAYMENT_KEYWORDS))
    off_platform_hits = len(find_keyword_matches_fuzzy(text, OFF_PLATFORM_KEYWORDS))
    credential_hits = len(find_keyword_matches_fuzzy(text, CREDENTIAL_KEYWORDS))

    letters = [c for c in text if c.isalpha()]
    capital_ratio = sum(1 for c in letters if c.isupper()) / len(letters) if letters else 0.0
    digit_count = sum(1 for c in text if c.isdigit())

    return {
        "urgency_keyword_count": urgency_keyword_hits,
        "guaranteed_return_keyword_count": guaranteed_return_hits,
        "countdown_phrase_count": countdown_hits,
        "exclamation_count": exclamation_count,
        "urgency_score": urgency_score,
        "has_wallet_address": int(bool(wallet_matches)),
        "has_url": int(bool(url_matches)),
        "url_count": len(url_matches),
        "has_email": int(bool(email_matches)),
        "has_phone_number": int(bool(phone_matches)),
        "payment_keyword_count": payment_hits,
        "off_platform_keyword_count": off_platform_hits,
        "credential_keyword_count": credential_hits,
        "capital_letter_ratio": round(capital_ratio, 4),
        "has_numeric_content": int(digit_count > 0),
    }

ENGINEERED_COLS = [
    "urgency_keyword_count", "guaranteed_return_keyword_count", "countdown_phrase_count",
    "exclamation_count", "urgency_score", "has_wallet_address",
    "has_url", "url_count", "has_email", "has_phone_number", "payment_keyword_count",
    "off_platform_keyword_count", "credential_keyword_count",
    "capital_letter_ratio", "has_numeric_content",
]

# Load, clean, engineer, split
df = pd.read_csv(input_path)
expected_columns = {"id", "platform", "text", "label"}
missing = expected_columns - set(df.columns)
if missing:
    raise ValueError(f"Dataset is missing expected columns: {missing}")

df["clean_text"] = df["text"].apply(clean_text)

engineered = pd.DataFrame([extract_engineered_features(t) for t in df["text"]], index=df.index)
df = pd.concat([df, engineered], axis=1)

train_df, test_df = train_test_split(
    df,
    test_size=args.test_size,
    random_state=args.random_state,
    stratify=df["label"],
)

print(f"Train: {train_df.shape[0]} rows | Test: {test_df.shape[0]} rows")

# TF-IDF, fit on train text only, never on test
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2)
vectorizer.fit(train_df["clean_text"])

X_train_tfidf = vectorizer.transform(train_df["clean_text"])
X_test_tfidf = vectorizer.transform(test_df["clean_text"])

y_train = (train_df["label"] == SCAM_LABEL).astype(int)
y_test = (test_df["label"] == SCAM_LABEL).astype(int)

print(f"TF-IDF vocabulary size: {len(vectorizer.get_feature_names_out())}")

# Save TF-IDF as sparse .npz (a dense CSV would be hundreds of MB); everything else as CSV
sp.save_npz(f"{output_dir}/train_tfidf.npz", X_train_tfidf)
sp.save_npz(f"{output_dir}/test_tfidf.npz", X_test_tfidf)

train_df[ENGINEERED_COLS].to_csv(f"{output_dir}/train_engineered.csv", index=False)
test_df[ENGINEERED_COLS].to_csv(f"{output_dir}/test_engineered.csv", index=False)

y_train.to_csv(f"{output_dir}/train_labels.csv", index=False, header=True)
y_test.to_csv(f"{output_dir}/test_labels.csv", index=False, header=True)

preprocessor = {
    "vectorizer": vectorizer,
    "engineered_cols": ENGINEERED_COLS,
    "feature_columns": list(vectorizer.get_feature_names_out()) + ENGINEERED_COLS,
}
joblib.dump(preprocessor, f"{output_dir}/preprocessor.joblib")

print("Preprocessing complete. Saved TF-IDF (.npz), engineered features, labels, and preprocessor.joblib.")
print(f"Final feature count: {len(preprocessor['feature_columns'])}")

Overwriting src/preprocess.py


In [37]:
%%writefile src/train.py
"""SageMaker Training Job.

Trains a Logistic Regression or Random Forest classifier on TF-IDF plus
engineered features, using the model type and hyperparameters passed in as
pipeline parameters. Saves a deployment bundle
containing the trained model and the fitted TF-IDF vectorizer from
preprocessor.joblib, so the endpoint can reproduce training-time feature
engineering. MLflow logging happens later, from the notebook, outside this
container.
"""
import os
import argparse
import pickle
import joblib
import pandas as pd
import scipy.sparse as sp

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    precision_score,
    recall_score,
)

parser = argparse.ArgumentParser()
parser.add_argument('--model-type', type=str, default='logistic_regression', choices=['logistic_regression', 'random_forest'])
parser.add_argument('--reg-c', type=float, default=10.0)
parser.add_argument('--penalty', type=str, default='l2')
parser.add_argument('--solver', type=str, default='lbfgs')
parser.add_argument('--max-iter', type=int, default=1000)
parser.add_argument('--n-estimators', type=int, default=200)
parser.add_argument('--max-depth', type=int, default=8)
parser.add_argument('--min-samples-leaf', type=int, default=2)
parser.add_argument('--random-state', type=int, default=42)

parser.add_argument('--team-id', type=str, default=os.environ.get('TEAM_ID', 'unknown-team'))
parser.add_argument('--student-id', type=str, default=os.environ.get('STUDENT_ID', 's000'))
parser.add_argument('--semester', type=str, default=os.environ.get('SEMESTER', '26S1'))
parser.add_argument('--run-name', type=str, default='sagemaker_pipeline_run')

parser.add_argument(
    '--model-dir',
    type=str,
    default=os.environ.get('SM_MODEL_DIR', '/opt/ml/model')
)
parser.add_argument(
    '--train',
    type=str,
    default=os.environ.get('SM_CHANNEL_TRAIN', '/opt/ml/input/data/train')
)
parser.add_argument(
    '--test',
    type=str,
    default=os.environ.get('SM_CHANNEL_TEST', '/opt/ml/input/data/test')
)
args = parser.parse_args()

os.makedirs(args.model_dir, exist_ok=True)

print('=== SageMaker Training Environment ===')
print(f'Train channel: {args.train}')
print(f'Test channel: {args.test}')
print(f'Model directory: {args.model_dir}')
print(f'Model type: {args.model_type}')

X_train_tfidf = sp.load_npz(os.path.join(args.train, 'train_tfidf.npz'))
X_test_tfidf = sp.load_npz(os.path.join(args.test, 'test_tfidf.npz'))

train_engineered = pd.read_csv(os.path.join(args.train, 'train_engineered.csv'))
test_engineered = pd.read_csv(os.path.join(args.test, 'test_engineered.csv'))

y_train = pd.read_csv(os.path.join(args.train, 'train_labels.csv')).squeeze('columns')
y_test = pd.read_csv(os.path.join(args.test, 'test_labels.csv')).squeeze('columns')

preprocessor_path = os.path.join(args.train, 'preprocessor.joblib')
if not os.path.exists(preprocessor_path):
    raise FileNotFoundError(
        f'preprocessor.joblib was not found at {preprocessor_path}. '
        'Rerun the ProcessingStep with the updated preprocess.py.'
    )

preprocessor = joblib.load(preprocessor_path)
engineered_cols = preprocessor['engineered_cols']

X_train = sp.hstack([X_train_tfidf, train_engineered[engineered_cols].values]).tocsr()
X_test = sp.hstack([X_test_tfidf, test_engineered[engineered_cols].values]).tocsr()

print(f'Train: {X_train.shape[0]} rows, {X_train.shape[1]} features')
print(f'Test: {X_test.shape[0]} rows')

if len(pd.Series(y_train).unique()) < 2:
    raise ValueError('Training labels contain fewer than two classes.')

if args.model_type == 'random_forest':
    model = RandomForestClassifier(
        n_estimators=args.n_estimators,
        max_depth=args.max_depth,
        min_samples_leaf=args.min_samples_leaf,
        class_weight='balanced',
        random_state=args.random_state,
        n_jobs=-1,
    )
else:
    model = LogisticRegression(
        C=args.reg_c,
        penalty=args.penalty,
        solver=args.solver,
        max_iter=args.max_iter,
        class_weight='balanced',
        random_state=args.random_state,
    )
model.fit(X_train, y_train)

all_metrics = {}
for split, X, y in [('train', X_train, y_train), ('test', X_test, y_test)]:
    predictions = model.predict(X)
    probabilities = model.predict_proba(X)[:, 1]

    all_metrics.update({
        f'{split}_accuracy': round(accuracy_score(y, predictions), 4),
        f'{split}_f1': round(f1_score(y, predictions, zero_division=0), 4),
        f'{split}_precision': round(
            precision_score(y, predictions, zero_division=0), 4
        ),
        f'{split}_recall': round(
            recall_score(y, predictions, zero_division=0), 4
        ),
    })

    if len(pd.Series(y).unique()) >= 2:
        all_metrics[f'{split}_auc_roc'] = round(
            roc_auc_score(y, probabilities), 4
        )
    else:
        all_metrics[f'{split}_auc_roc'] = None
        print(f'Warning: {split} split has only one class; AUC-ROC unavailable.')

print('=== Metrics ===')
for metric_name, metric_value in all_metrics.items():
    print(f'{metric_name}: {metric_value}')

model_bundle = {
    'model': model,
    'preprocessor': preprocessor,
    'engineered_cols': engineered_cols,
    'input_format': 'raw_text_json',
    'description': (
        'Deployment bundle containing trained model and fitted TF-IDF vectorizer. '
        'Endpoint accepts raw message text as JSON: {"text": "..."}'
    ),
}

joblib.dump(model_bundle, os.path.join(args.model_dir, 'model.joblib'))

with open(os.path.join(args.model_dir, 'model.pkl'), 'wb') as f:
    pickle.dump(model_bundle, f)

print(f"Model bundle saved: {os.path.join(args.model_dir, 'model.joblib')}")
print(f"Legacy bundle saved: {os.path.join(args.model_dir, 'model.pkl')}")

if all_metrics['test_auc_roc'] is None:
    raise ValueError('Test AUC-ROC is unavailable; cannot evaluate the quality gate.')

print(f"Test AUC-ROC: {all_metrics['test_auc_roc']}")
print(f"test_accuracy: {all_metrics['test_accuracy']}")
print(f"test_f1: {all_metrics['test_f1']}")

Overwriting src/train.py


In [38]:
%%writefile src/inference.py
"""SageMaker inference handler for the deployed crypto-scam-detector endpoint.

Accepts JSON input with raw message text, e.g. {"text": "..."}, a list of
such records, or {"instances": [...]}. Applies the same text cleaning,
engineered-feature extraction and TF-IDF transform saved in model.joblib
before predicting, matching utils/preprocessing.py, so serving matches
training.
"""
import os
import re
import json
import pickle
import joblib
import pandas as pd
import scipy.sparse as sp

import subprocess
import sys

try:
    from rapidfuzz import fuzz
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "rapidfuzz"])
    from rapidfuzz import fuzz

URL_PATTERN = re.compile(r"https?://[^\s]+|www\.[^\s]+")

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.strip()
    text = URL_PATTERN.sub(" <url> ", text)
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()

URGENT_KEYWORDS = [
    "urgent", "immediately", "hurry", "act now", "act fast", "limited time",
    "limited slots", "limited spots", "today only", "last chance",
    "final notice", "final warning", "don't miss", "don't wait",
    "expires", "expiring", "closing soon", "ending soon", "before it's too late",
    "respond now", "reply now", "confirm now", "verify now", "claim now",
    "while supplies last", "only a few left", "act before", "time-sensitive",
    "this offer", "exclusive offer", "one time offer",
]
GUARANTEED_RETURN_KEYWORDS = [
    "guaranteed", "guarantee", "risk-free", "risk free", "no risk",
    "zero risk", "sure profit", "sure win", "can't lose", "cannot lose",
    "double your money", "triple your money", "multiply your", "10x", "100x",
    "passive income", "easy money", "get rich", "financial freedom",
    "life-changing", "once in a lifetime", "secret method", "proven strategy",
    "insider information", "insider tip", "exclusive access", "vip access",
]
PAYMENT_KEYWORDS = [
    "deposit", "transfer", "send funds", "send payment", "send money",
    "top up", "top-up", "processing fee", "activation fee", "unlock fee",
    "advance fee", "small fee", "gas fee", "network fee", "wallet",
    "crypto", "bitcoin", "ethereum", "usdt", "stablecoin",
]
OFF_PLATFORM_KEYWORDS = [
    "telegram", "whatsapp", "discord", "signal", "wechat", "line app",
    "private chat", "private message", "dm me", "direct message",
    "text me", "call me", "contact me directly", "reach out privately",
    "add me on",
]
CREDENTIAL_KEYWORDS = [
    "seed phrase", "private key", "recovery phrase", "recovery key",
    "wallet password", "login credentials", "verification code", "otp",
    "security code", "pin number", "social security"
]

WALLET_PATTERN = re.compile(r"\b(?:0x[a-fA-F0-9]{40}|[13][a-km-zA-HJ-NP-Z1-9]{25,34})\b")
EMAIL_PATTERN = re.compile(r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}")
PHONE_PATTERN = re.compile(r"(?:\+?\d[\d\s-]{7,}\d)")
COUNTDOWN_PATTERN = re.compile(r"\b\d+\s*(?:hour|hours|hr|hrs|minute|minutes|min|mins|day|days)\b", re.IGNORECASE)

def find_keyword_matches_fuzzy(message, keywords, threshold=85):
    message_lower = message.lower()
    tokens = message_lower.split()
    matches = []
    for kw in keywords:
        kw_lower = kw.lower()
        if " " in kw_lower:
            if fuzz.partial_ratio(kw_lower, message_lower) >= threshold:
                matches.append(kw)
        else:
            if any(fuzz.ratio(kw_lower, tok) >= threshold for tok in tokens):
                matches.append(kw)
    return matches

def extract_engineered_features(text):
    if not isinstance(text, str):
        text = ""

    urgency_keyword_hits = len(find_keyword_matches_fuzzy(text, URGENT_KEYWORDS))
    guaranteed_return_hits = len(find_keyword_matches_fuzzy(text, GUARANTEED_RETURN_KEYWORDS))
    countdown_hits = len(COUNTDOWN_PATTERN.findall(text))
    exclamation_count = text.count("!")
    urgency_score = urgency_keyword_hits + countdown_hits + min(exclamation_count, 3)

    wallet_matches = WALLET_PATTERN.findall(text)
    url_matches = URL_PATTERN.findall(text)
    email_matches = EMAIL_PATTERN.findall(text)
    phone_matches = PHONE_PATTERN.findall(text)
    payment_hits = len(find_keyword_matches_fuzzy(text, PAYMENT_KEYWORDS))
    off_platform_hits = len(find_keyword_matches_fuzzy(text, OFF_PLATFORM_KEYWORDS))
    credential_hits = len(find_keyword_matches_fuzzy(text, CREDENTIAL_KEYWORDS))

    letters = [c for c in text if c.isalpha()]
    capital_ratio = sum(1 for c in letters if c.isupper()) / len(letters) if letters else 0.0
    digit_count = sum(1 for c in text if c.isdigit())

    return {
        "urgency_keyword_count": urgency_keyword_hits,
        "guaranteed_return_keyword_count": guaranteed_return_hits,
        "countdown_phrase_count": countdown_hits,
        "exclamation_count": exclamation_count,
        "urgency_score": urgency_score,
        "has_wallet_address": int(bool(wallet_matches)),
        "has_url": int(bool(url_matches)),
        "url_count": len(url_matches),
        "has_email": int(bool(email_matches)),
        "has_phone_number": int(bool(phone_matches)),
        "payment_keyword_count": payment_hits,
        "off_platform_keyword_count": off_platform_hits,
        "credential_keyword_count": credential_hits,
        "capital_letter_ratio": round(capital_ratio, 4),
        "has_numeric_content": int(digit_count > 0),
    }


def model_fn(model_dir):
    """Load the model bundle from the SageMaker model directory."""
    joblib_path = os.path.join(model_dir, 'model.joblib')
    pkl_path = os.path.join(model_dir, 'model.pkl')

    if os.path.exists(joblib_path):
        bundle = joblib.load(joblib_path)
    elif os.path.exists(pkl_path):
        with open(pkl_path, 'rb') as f:
            bundle = pickle.load(f)
    else:
        raise FileNotFoundError('Neither model.joblib nor model.pkl was found.')

    return bundle


def input_fn(body, content_type='application/json'):
    """Parse JSON request body into a DataFrame of raw message-text records."""
    if content_type != 'application/json':
        raise ValueError(f'Unsupported content type: {content_type}')

    payload = json.loads(body)

    if isinstance(payload, dict):
        if 'instances' in payload:
            payload = payload['instances']
        else:
            payload = [payload]

    if not isinstance(payload, list):
        raise ValueError('JSON input must be a dictionary, a list of dictionaries, or {"instances": [...]}')

    return pd.DataFrame(payload)


def get_feature_names(bundle):
    """Combine TF-IDF vocabulary with engineered feature names, in the same order
    used by the sp.hstack() call in predict_fn, so model coefficients can be
    matched back to human-readable feature names for explanations."""
    preprocessor = bundle['preprocessor']
    vectorizer = preprocessor['vectorizer']
    engineered_cols = preprocessor['engineered_cols']
    tfidf_names = vectorizer.get_feature_names_out().tolist()
    return tfidf_names + list(engineered_cols)


def top_contributing_features(x_row, coef, feature_names, top_n=5):
    """Coefficient-based explanation for a single row: contribution = feature
    value * coefficient. This is a lightweight linear-model approximation, not
    a SHAP or LIME-style explanation, but is cheap enough to run on every
    request and gives a directionally useful signal for which words/patterns
    pushed the prediction toward scam vs legitimate."""
    row = x_row.toarray().ravel() if sp.issparse(x_row) else x_row.ravel()
    contributions = row * coef
    nonzero_idx = contributions.nonzero()[0]
    if len(nonzero_idx) == 0:
        return []
    ranked = sorted(nonzero_idx, key=lambda i: abs(contributions[i]), reverse=True)[:top_n]
    return [
        {
            'feature': feature_names[i],
            'contribution': round(float(contributions[i]), 4),
            'direction': 'scam' if contributions[i] > 0 else 'legitimate',
        }
        for i in ranked
    ]


def predict_fn(data, bundle):
    """Clean text, extract engineered features, TF-IDF transform, then predict.
    Also computes a top-features explanation for each row when the model
    exposes coefficients (e.g. logistic regression)."""
    if 'text' not in data.columns:
        raise ValueError('Input must include a "text" field with the raw message.')

    model = bundle['model']
    preprocessor = bundle['preprocessor']
    vectorizer = preprocessor['vectorizer']
    engineered_cols = preprocessor['engineered_cols']

    clean = data['text'].apply(clean_text)
    engineered = pd.DataFrame(
        [extract_engineered_features(t) for t in data['text']], index=data.index
    )

    X_tfidf = vectorizer.transform(clean)
    X = sp.hstack([X_tfidf, engineered[engineered_cols].values]).tocsr()

    predictions = model.predict(X)
    probabilities = model.predict_proba(X)[:, 1]

    explanations = None
    if hasattr(model, 'coef_'):
        feature_names = get_feature_names(bundle)
        coef = model.coef_.ravel()
        explanations = [
            top_contributing_features(X[i], coef, feature_names)
            for i in range(X.shape[0])
        ]

    return predictions, probabilities, explanations


def output_fn(prediction, accept='application/json'):
    preds, probas, explanations = prediction
    if explanations is None:
        explanations = [[] for _ in preds]
    response = [
        {
            'prediction': int(p),
            'label': 'Scam' if int(p) == 1 else 'Legitimate',
            'probability': round(float(b), 4),
            'top_features': feats,
        }
        for p, b, feats in zip(preds, probas, explanations)
    ]
    return json.dumps(response), accept


Overwriting src/inference.py


In [39]:
print("Scripts written:")
for fn in ["preprocess.py", "train.py", "inference.py"]:
    size = os.path.getsize(f"src/{fn}")
    print(f"  src/{fn}  ({size} bytes)")

Scripts written:
  src/preprocess.py  (8966 bytes)
  src/train.py  (6145 bytes)
  src/inference.py  (9767 bytes)


## 1A. Upload pipeline source files to S3

No requirements_train.txt is needed since the training container doesn't use MLflow. preprocess.py and train.py save a fitted TF-IDF vectorizer and engineered feature columns instead of a StandardScaler, since this is text classification rather than tabular data, and the endpoint reuses this logic on raw text input.


In [40]:
from pathlib import Path

s3_client = boto3.client("s3")

SOURCE_DIR = Path("src")
FILES_TO_UPLOAD = [
    "preprocess.py",
    "train.py",
    "inference.py",
]

for filename in FILES_TO_UPLOAD:
    local_path = SOURCE_DIR / filename
    s3_key = f"{SCRIPTS_S3_PREFIX}/{filename}"

    if not local_path.exists():
        raise FileNotFoundError(f"Missing local source file: {local_path}")

    s3_client.upload_file(str(local_path), BUCKET, s3_key)
    print(f"Uploaded {local_path} -> s3://{BUCKET}/{s3_key}")

print("Pipeline source files uploaded to:", SCRIPTS_S3_URI)

Uploaded src/preprocess.py -> s3://nyp-26s1-iti113/iti113/team03/data/crypto-scam-detector/pipeline_src/preprocess.py
Uploaded src/train.py -> s3://nyp-26s1-iti113/iti113/team03/data/crypto-scam-detector/pipeline_src/train.py
Uploaded src/inference.py -> s3://nyp-26s1-iti113/iti113/team03/data/crypto-scam-detector/pipeline_src/inference.py
Pipeline source files uploaded to: s3://nyp-26s1-iti113/iti113/team03/data/crypto-scam-detector/pipeline_src


## 1B. Download pipeline source files from S3

Recreates the local pipeline_src folder by downloading from S3, confirming the pipeline uses the S3-preplaced files rather than the original notebook-generated src files.


In [41]:
import shutil

local_src = Path(LOCAL_PIPELINE_SRC)

if local_src.exists():
    shutil.rmtree(local_src)

local_src.mkdir(parents=True, exist_ok=True)

for filename in FILES_TO_UPLOAD:
    s3_key = f"{SCRIPTS_S3_PREFIX}/{filename}"
    local_path = local_src / filename

    s3_client.download_file(BUCKET, s3_key, str(local_path))
    print(f"Downloaded s3://{BUCKET}/{s3_key} -> {local_path}")

print("Downloaded files:")
for p in sorted(local_src.iterdir()):
    print("-", p)

Downloaded s3://nyp-26s1-iti113/iti113/team03/data/crypto-scam-detector/pipeline_src/preprocess.py -> pipeline_src/preprocess.py
Downloaded s3://nyp-26s1-iti113/iti113/team03/data/crypto-scam-detector/pipeline_src/train.py -> pipeline_src/train.py
Downloaded s3://nyp-26s1-iti113/iti113/team03/data/crypto-scam-detector/pipeline_src/inference.py -> pipeline_src/inference.py
Downloaded files:
- pipeline_src/inference.py
- pipeline_src/preprocess.py
- pipeline_src/train.py


## 2. Define the SageMaker Pipeline

Four SageMaker Pipeline steps:

1. ProcessingStep: runs preprocess.py, outputs train/test CSVs and preprocessor.joblib to S3.
2. TrainingStep: runs train.py, trains the model, loads preprocessor.joblib, and saves a model.joblib deployment bundle.
3. ConditionStep: checks AUC is greater than or equal to the threshold before allowing registration.
4. ModelStep: registers the model in SageMaker Model Registry with status PendingManualApproval.


In [42]:
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.steps import ProcessingStep, TrainingStep
from sagemaker.workflow.conditions import ConditionGreaterThanOrEqualTo
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.workflow.functions import JsonGet
from sagemaker.workflow.parameters import ParameterFloat, ParameterInteger, ParameterString
from sagemaker.workflow.model_step import ModelStep
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.sklearn.estimator import SKLearn
from sagemaker.processing import ProcessingInput, ProcessingOutput
from sagemaker.model import Model
from sagemaker.workflow.pipeline_context import PipelineSession

pipeline_session = PipelineSession()

# Pipeline parameters default to the model type and hyperparameters resolved above
p_model_type = ParameterString(name='ModelType', default_value=MODEL_TYPE)
# --reg-c not --C: SageMaker turns single-character hyperparameter keys into a short flag that argparse rejects
p_c = ParameterFloat(name='RegularizationC', default_value=MODEL_HYPERPARAMS.get('C', 10.0))
p_penalty = ParameterString(name='Penalty', default_value=MODEL_HYPERPARAMS.get('penalty', 'l2'))
p_solver = ParameterString(name='Solver', default_value=MODEL_HYPERPARAMS.get('solver', 'lbfgs'))
p_max_iter = ParameterInteger(name='MaxIter', default_value=MODEL_HYPERPARAMS.get('max_iter', 1000))
p_n_est = ParameterInteger(name='NEstimators', default_value=MODEL_HYPERPARAMS.get('n_estimators', 200))
p_depth = ParameterInteger(name='MaxDepth', default_value=MODEL_HYPERPARAMS.get('max_depth', 8))
p_samples = ParameterInteger(name='MinSamplesLeaf', default_value=MODEL_HYPERPARAMS.get('min_samples_leaf', 2))
p_gate = ParameterFloat(name='QualityGateAUC', default_value=QUALITY_GATE_AUC)

print('Pipeline parameters defined.')


Pipeline parameters defined.


In [43]:
# Defines the SKLearnProcessor and ProcessingStep that runs preprocess.py
processor = SKLearnProcessor(
    framework_version='1.2-1', instance_type=PROCESSING_INSTANCE_TYPE,
    instance_count=1, role=role, sagemaker_session=pipeline_session,
    base_job_name=f'iti113-{TEAM_ID}-{STUDENT_ID}-process')

step_process = ProcessingStep(
    name='PreprocessData',
    processor=processor,
    inputs=[ProcessingInput(source=RAW_DATA_URI,
                            destination='/opt/ml/processing/input')],  # lands at input/<dataset filename>
    outputs=[ProcessingOutput(output_name='processed',
                              source='/opt/ml/processing/output',
                              destination=f'{PIPELINE_ROOT}/processed')],
    code=f'{LOCAL_PIPELINE_SRC}/preprocess.py',
    job_arguments=['--test-size','0.2','--random-state','42']
)
print('Step 1 (ProcessingStep) defined.')

Step 1 (ProcessingStep) defined.


In [44]:
# Defines the SKLearn estimator and TrainingStep that runs train.py
# Training container has no Databricks host, token or MLflow dependency; it only trains and prints metrics
estimator = SKLearn(
    entry_point="train.py",
    source_dir=LOCAL_PIPELINE_SRC,
    framework_version="1.2-1",
    instance_type=TRAINING_INSTANCE_TYPE,
    instance_count=1,
    role=role,
    base_job_name=f"iti113-{TEAM_ID}-{STUDENT_ID}-train",
    sagemaker_session=pipeline_session,
    hyperparameters={
        "model-type": p_model_type,
        "reg-c": p_c,
        "penalty": p_penalty,
        "solver": p_solver,
        "max-iter": p_max_iter,
        "n-estimators": p_n_est,
        "max-depth": p_depth,
        "min-samples-leaf": p_samples,
        "random-state": 42,
        "team-id": TEAM_ID,
        "student-id": STUDENT_ID,
        "semester": SEMESTER,
        "run-name": "sagemaker_pipeline_run",
    },
    environment={
        "TEAM_ID": TEAM_ID,
        "STUDENT_ID": STUDENT_ID,
        "SEMESTER": SEMESTER,
    },
    metric_definitions=[
        {"Name": "test_auc_roc", "Regex": "Test AUC-ROC: ([0-9\\.]+)"},
        {"Name": "test_accuracy", "Regex": "test_accuracy: ([0-9\\.]+)"},
        {"Name": "test_f1", "Regex": "test_f1: ([0-9\\.]+)"},
    ],
    tags=[
        {"Key": "Course", "Value": "ITI113"},
        {"Key": "Semester", "Value": SEMESTER},
        {"Key": "Team", "Value": TEAM_ID},
        {"Key": "Student", "Value": STUDENT_ID},
    ],
)

processed_uri = step_process.properties.ProcessingOutputConfig.Outputs["processed"].S3Output.S3Uri

step_train = TrainingStep(
    name="TrainModel",
    estimator=estimator,
    inputs={
        # No content_type set: Processing output mixes .npz and .csv files, and train.py reads each directly
        "train": sagemaker.inputs.TrainingInput(s3_data=processed_uri),
        "test": sagemaker.inputs.TrainingInput(s3_data=processed_uri),
    },
)

print("Step 2 (TrainingStep) defined.")

Step 2 (TrainingStep) defined.


In [45]:
# Defines the Model and ModelStep that registers it in SageMaker Model Registry
model = Model(
    image_uri=estimator.training_image_uri(region),
    model_data=step_train.properties.ModelArtifacts.S3ModelArtifacts,
    sagemaker_session=pipeline_session,
    role=role,
    entry_point='inference.py',
    source_dir=LOCAL_PIPELINE_SRC
)
step_register = ModelStep(
    name='RegisterModel',
    step_args=model.register(
        content_types=['application/json'],
        response_types=['application/json'],
        inference_instances=['ml.m5.large'],
        transform_instances=['ml.m5.large'],
        model_package_group_name=MODEL_PACKAGE_GROUP,
        approval_status='PendingManualApproval',
    )
)
print('Step 3 (ModelStep) defined.')

Step 3 (ModelStep) defined.


In [46]:
# Gates registration on the SageMaker-captured test AUC (test_auc_roc from metric_definitions).
# A failed gate is an explicit FailStep rather than a silent no-op, so the pipeline execution
# itself reports Failed (visible in Studio and via describe_pipeline_execution) when the quality
# bar isn't met, instead of quietly finishing with nothing registered.
from sagemaker.workflow.fail_step import FailStep
from sagemaker.workflow.functions import Join

condition = ConditionGreaterThanOrEqualTo(
    left=step_train.properties.FinalMetricDataList["test_auc_roc"].Value,
    right=p_gate
)

step_fail = FailStep(
    name="QualityGateFailed",
    error_message=Join(
        on=" ",
        values=[
            "Model failed the quality gate: test AUC-ROC",
            step_train.properties.FinalMetricDataList["test_auc_roc"].Value,
            "is below the required threshold",
            p_gate,
        ],
    ),
)

step_condition = ConditionStep(
    name="AUCQualityGate",
    conditions=[condition],
    if_steps=[step_register],
    else_steps=[step_fail]
)
print("Step 4 (ConditionStep) defined, with an explicit FailStep on the else branch.")


Step 4 (ConditionStep) defined, with an explicit FailStep on the else branch.


In [47]:
# Assemble and upsert the pipeline
pipeline = Pipeline(
    name=PIPELINE_NAME,
    parameters=[p_model_type, p_c, p_penalty, p_solver, p_max_iter, p_n_est, p_depth, p_samples, p_gate],
    steps=[step_process, step_train, step_condition],
    sagemaker_session=pipeline_session
)
pipeline.upsert(role_arn=role)
print(f'Pipeline "{PIPELINE_NAME}" upserted.')

Pipeline "iti113-team03-crypto-scam-detector" upserted.


## 3. Execute the Pipeline

Starts a pipeline execution with the default hyperparameters and quality gate, then polls step status until the run finishes.


In [48]:
execution = pipeline.start(
    execution_display_name="champion-c10-run",
    execution_description="Champion: tuned logistic regression, C=10.0",
    parameters={
        'ModelType': MODEL_TYPE,
        'RegularizationC': MODEL_HYPERPARAMS.get('C', 10.0),
        'Penalty': MODEL_HYPERPARAMS.get('penalty', 'l2'),
        'Solver': MODEL_HYPERPARAMS.get('solver', 'lbfgs'),
        'MaxIter': MODEL_HYPERPARAMS.get('max_iter', 1000),
        'NEstimators': MODEL_HYPERPARAMS.get('n_estimators', 200),
        'MaxDepth': MODEL_HYPERPARAMS.get('max_depth', 8),
        'MinSamplesLeaf': MODEL_HYPERPARAMS.get('min_samples_leaf', 2),
        'QualityGateAUC': 0.85,
    },
)
print(f'Execution ARN: {execution.arn}')

Execution ARN: arn:aws:sagemaker:ap-southeast-1:044528205969:pipeline/iti113-team03-crypto-scam-detector/execution/0msdw4kjcdno


In [49]:
import time

prev = {}

while True:

    desc = execution.describe()
    status = desc["PipelineExecutionStatus"]

    steps = execution.list_steps()

    if isinstance(steps, dict):
        steps = steps.get("PipelineExecutionSteps", [])

    for step in steps:
        n = step["StepName"]
        s = step["StepStatus"]

        if prev.get(n) != s:
            print(f"{n:<20} {s}")
            prev[n] = s

    if status in ("Succeeded", "Failed", "Stopped"):
        print(f"\nPipeline Status: {status}")
        break

    time.sleep(30)


PreprocessData       Executing
TrainModel           Executing
PreprocessData       Succeeded
RegisterModel-RepackModel-0 Executing
AUCQualityGate       Succeeded
TrainModel           Succeeded
RegisterModel-RegisterModel Succeeded
RegisterModel-RepackModel-0 Succeeded

Pipeline Status: Succeeded


## 4. Log the Completed SageMaker Run to the SageMaker MLflow App

Reads the completed SageMaker training job for its captured metrics, hyperparameters, job name, pipeline execution ARN and model-artifact S3 URI, then logs these as an MLflow run in the team's SageMaker Serverless MLflow App experiment. 

In [50]:
# Get_pipeline_steps handles both list- and dict-returning SDK versions
import mlflow

def get_pipeline_steps(execution):
    response = execution.list_steps()
    if isinstance(response, list):
        return response
    return response.get("PipelineExecutionSteps", [])


if execution.describe()["PipelineExecutionStatus"] != "Succeeded":
    raise RuntimeError(
        "The SageMaker Pipeline has not succeeded. "
        "Resolve pipeline failures before logging to MLflow."
    )

steps = get_pipeline_steps(execution)

print("Pipeline steps:")
for step in steps:
    print(f"  {step['StepName']}: {step['StepStatus']}")

train_step_info = next(
    (
        step for step in steps
        if step["StepName"] == "TrainModel"
        and step["StepStatus"] == "Succeeded"
    ),
    None
)

if train_step_info is None:
    raise RuntimeError(
        "A successful TrainModel step was not found in this pipeline execution."
    )

training_job_arn = train_step_info["Metadata"]["TrainingJob"]["Arn"]
training_job_name = training_job_arn.rsplit("/", 1)[-1]

sm_client = boto3.client("sagemaker", region_name=region)
training_job = sm_client.describe_training_job(
    TrainingJobName=training_job_name
)

# SageMaker captures the metrics printed by train.py through metric_definitions.
captured_metrics = {
    item["MetricName"]: float(item["Value"])
    for item in training_job.get("FinalMetricDataList", [])
    if item["MetricName"] in {"test_auc_roc", "test_accuracy", "test_f1"}
}

if not captured_metrics:
    raise RuntimeError(
        "No captured SageMaker metrics were found. "
        "Check train.py output and estimator.metric_definitions."
    )

model_artifact_s3_uri = training_job["ModelArtifacts"]["S3ModelArtifacts"]
training_hyperparameters = training_job.get("HyperParameters", {})

print(
    f"Training job: {training_job_name}\n"
    f"Model artefact: {model_artifact_s3_uri}\n"
    f"Captured metrics: {captured_metrics}"
)

# Log to SageMaker Serverless MLflow App.
mlflow.set_tracking_uri(MLFLOW_APP_ARN)
experiment = mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

mlflow_run_name = (
    f"{TEAM_ID}_{STUDENT_ID}_sagemaker_pipeline_"
    f"{int(time.time())}"
)

with mlflow.start_run(run_name=mlflow_run_name) as run:
    mlflow.set_tags({
        "course": "ITI113",
        "semester": SEMESTER,
        "team_id": TEAM_ID,
        "student_id": STUDENT_ID,
        "dataset": "crypto-scam-detector",
        "execution_environment": "aws_sagemaker_pipeline",
        "tracking_backend": "sagemaker_mlflow_app",
        "pipeline_execution_arn": execution.arn,
        "training_job_name": training_job_name,
        "sagemaker_model_artifact_s3_uri": model_artifact_s3_uri,
        "mlflow_app_arn": MLFLOW_APP_ARN,
        "mlflow_experiment": MLFLOW_EXPERIMENT_NAME,
    })

    # Hyperparameters arrive from SageMaker as strings, which are valid MLflow params.
    mlflow.log_params(training_hyperparameters)
    mlflow.log_metrics(captured_metrics)

    # Store a small, portable traceability record as an MLflow artefact.
    run_summary = {
        "pipeline_execution_arn": execution.arn,
        "training_job_name": training_job_name,
        "training_job_arn": training_job_arn,
        "model_artifact_s3_uri": model_artifact_s3_uri,
        "metrics": captured_metrics,
        "hyperparameters": training_hyperparameters,
        "team_id": TEAM_ID,
        "student_id": STUDENT_ID,
        "semester": SEMESTER,
        "mlflow_app_arn": MLFLOW_APP_ARN,
        "mlflow_experiment": MLFLOW_EXPERIMENT_NAME,
        "tracking_backend": "sagemaker_mlflow_app",
    }

    summary_file = "sagemaker_pipeline_run_summary.json"
    with open(summary_file, "w") as f:
        json.dump(run_summary, f, indent=2)

    mlflow.log_artifact(
        summary_file,
        artifact_path="sagemaker_pipeline"
    )

    mlflow_run_id = run.info.run_id
    mlflow_experiment_id = run.info.experiment_id

print("SageMaker MLflow App logging completed.")
print("MLflow run ID:", mlflow_run_id)
print("MLflow App ARN:", MLFLOW_APP_ARN)
print("Experiment:", MLFLOW_EXPERIMENT_NAME)
print("Experiment ID:", mlflow_experiment_id)

Pipeline steps:
  RegisterModel-RegisterModel: Succeeded
  RegisterModel-RepackModel-0: Succeeded
  AUCQualityGate: Succeeded
  TrainModel: Succeeded
  PreprocessData: Succeeded
Training job: pipelines-0msdw4kjcdno-TrainModel-2chexIqPE4
Model artefact: s3://sagemaker-ap-southeast-1-044528205969/pipelines-0msdw4kjcdno-TrainModel-2chexIqPE4/output/model.tar.gz
Captured metrics: {'test_auc_roc': 0.8899999856948853, 'test_accuracy': 0.7713000178337097, 'test_f1': 0.775600016117096}
🏃 View run team03_s301_sagemaker_pipeline_1787129360 at: https://mlflow.sagemaker.ap-southeast-1.app.aws/#/experiments/1/runs/fb062b6441f74c28ae7025cf49b2d539
🧪 View experiment at: https://mlflow.sagemaker.ap-southeast-1.app.aws/#/experiments/1
SageMaker MLflow App logging completed.
MLflow run ID: fb062b6441f74c28ae7025cf49b2d539
MLflow App ARN: arn:aws:sagemaker:ap-southeast-1:044528205969:mlflow-app/app-J5AYUG4AJHVW
Experiment: ITI113/team03/Experiment1
Experiment ID: 1


## 5. Deploy Serverless Endpoint

After the pipeline succeeds, the model sits in Model Registry with status PendingManualApproval. We approve it here, then deploy as a Serverless Endpoint


In [51]:
sm = boto3.client('sagemaker')

# Get the latest registered model package
pkgs = sm.list_model_packages(
    ModelPackageGroupName=MODEL_PACKAGE_GROUP,
    SortBy='CreationTime', SortOrder='Descending', MaxResults=1
)['ModelPackageSummaryList']

if not pkgs:
    print('No model packages found. Check the pipeline completed the Register step.')
else:
    pkg_arn = pkgs[0]['ModelPackageArn']
    print(f'Model package: {pkg_arn}\nStatus: {pkgs[0]["ModelApprovalStatus"]}')

Model package: arn:aws:sagemaker:ap-southeast-1:044528205969:model-package/team03-CryptoScamDetector/45
Status: PendingManualApproval


In [52]:
# Approve the model
sm.update_model_package(ModelPackageArn=pkg_arn, ModelApprovalStatus='Approved')
print(f'Approved: {pkg_arn}')

Approved: arn:aws:sagemaker:ap-southeast-1:044528205969:model-package/team03-CryptoScamDetector/45


In [53]:
from sagemaker import ModelPackage
from sagemaker.serverless import ServerlessInferenceConfig

sm = boto3.client('sagemaker')

# Clean up any leftover endpoint/config, create_endpoint_config fails if one exists
try:
    sm.delete_endpoint(EndpointName=ENDPOINT_NAME)
    print(f'Deleted existing endpoint: {ENDPOINT_NAME}')
except sm.exceptions.ClientError:
    pass

try:
    sm.delete_endpoint_config(EndpointConfigName=ENDPOINT_NAME)
    print(f'Deleted existing endpoint config: {ENDPOINT_NAME}')
except sm.exceptions.ClientError:
    pass

deployable = ModelPackage(
    role=role, model_package_arn=pkg_arn, sagemaker_session=sagemaker.Session())

serverless_cfg = ServerlessInferenceConfig(memory_size_in_mb=2048, max_concurrency=5)

print(f'Deploying serverless endpoint: {ENDPOINT_NAME}')
predictor = deployable.deploy(
    serverless_inference_config=serverless_cfg,
    endpoint_name=ENDPOINT_NAME
)
print(f'Endpoint ready: {ENDPOINT_NAME}')

Deleted existing endpoint: iti113-team03-crypto-scam-detector
Deleted existing endpoint config: iti113-team03-crypto-scam-detector
Deploying serverless endpoint: iti113-team03-crypto-scam-detector
-----!Endpoint ready: iti113-team03-crypto-scam-detector


## 6. Verify and Test the Live Endpoint

Confirms the champion endpoint is `InService`, traces it back to the Model Registry package it was approved from, then tests it with single and batch invocations.


### 6.1 Confirm the endpoint is InService

In [54]:
endpoint_desc = sm.describe_endpoint(EndpointName=ENDPOINT_NAME)
print(
    f"Endpoint name: {endpoint_desc['EndpointName']}\n"
    f"Status: {endpoint_desc['EndpointStatus']}\n"
    f"Creation time: {endpoint_desc['CreationTime']}\n"
    f"Last modified: {endpoint_desc['LastModifiedTime']}"
)

if endpoint_desc["EndpointStatus"] != "InService":
    print("\nEndpoint is not InService yet. Wait for deployment (Section 5) to finish before testing.")


Endpoint name: iti113-team03-crypto-scam-detector
Status: InService
Creation time: 2026-08-19 08:49:24.017000+00:00
Last modified: 2026-08-19 08:52:02.492000+00:00


### 6.2 Trace the endpoint back to its Model Registry package (optional)

dataset → preprocessing → training job → Model Registry package → endpoint.


In [55]:
endpoint_config_name = endpoint_desc["EndpointConfigName"]
endpoint_config = sm.describe_endpoint_config(EndpointConfigName=endpoint_config_name)

for variant in endpoint_config.get("ProductionVariants", []):
    variant_name = variant.get("VariantName")
    model_name = variant.get("ModelName")
    print(f"Variant name: {variant_name}\nModel name: {model_name}")

    model_desc = sm.describe_model(ModelName=model_name)
    containers = model_desc.get("Containers") or model_desc.get("PrimaryContainer")
    if isinstance(containers, dict):
        containers = [containers]

    for container in containers or []:
        model_package_arn = container.get("ModelPackageName")
        if model_package_arn:
            package_desc = sm.describe_model_package(ModelPackageName=model_package_arn)
            print(f"Model package: {model_package_arn}\nModel package status: {package_desc.get('ModelApprovalStatus')}")
        else:
            print(f"Model artifact (ModelDataUrl): {container.get('ModelDataUrl')}")


Variant name: AllTraffic
Model name: team03-CryptoScamDetector-2026-08-19-08-49-22-386
Model package: arn:aws:sagemaker:ap-southeast-1:044528205969:model-package/team03-CryptoScamDetector/45
Model package status: Approved


### 6.3 Test endpoint invocation

The endpoint accepts JSON input with the raw message text — the same shape `pages/1_Scam_Detector.py` sends via the relay (Notebook 06). It returns a list with one object containing `prediction`, `label` and `probability`. `inference.py` performs cleaning, engineered-feature extraction and the TF-IDF transform internally, so callers only ever send raw text.


In [56]:
runtime = boto3.client('sagemaker-runtime', region_name=region)


def invoke_scam_detector(text, endpoint_name=ENDPOINT_NAME):
    """Invoke the deployed crypto-scam-detector endpoint with a single raw message.

    This is the exact call the Notebook 06 relay makes on the champion model's behalf.
    """
    response = runtime.invoke_endpoint(
        EndpointName=endpoint_name,
        ContentType="application/json",
        Body=json.dumps({"text": text}),
    )
    return json.loads(response["Body"].read())[0]


scam_message = (
    "URGENT: Your wallet has been selected for a guaranteed 100% profit airdrop! "
    "Deposit 500 USDT to your wallet address within 1 hour to claim now. "
    "Contact us on Telegram immediately, don't miss out!"
)
legit_message = (
    "Been dollar-cost averaging into ETH for about a year now, curious what "
    "everyone's thoughts are on the current market conditions."
)
borderline_message = (
    "Hey, our community wallet is doing a small giveaway this week, check the "
    "pinned post in the group for details."
)

for label, msg in [
    ("SCAM-STYLE MESSAGE", scam_message),
    ("LEGIT-STYLE MESSAGE", legit_message),
    ("BORDERLINE / AMBIGUOUS MESSAGE", borderline_message),
]:
    result = invoke_scam_detector(msg)
    print(f"{label}\nPrediction: {result['label']}\nProbability: {result['probability']:.1%}\n")


SCAM-STYLE MESSAGE
Prediction: Scam
Probability: 99.9%

LEGIT-STYLE MESSAGE
Prediction: Legitimate
Probability: 10.5%

BORDERLINE / AMBIGUOUS MESSAGE
Prediction: Legitimate
Probability: 20.9%



### 6.4 Batch invocation

`inference.py` also accepts an `instances` list holding multiple text objects in a single request — useful for scoring several examples, or a small batch, at once.


In [57]:
batch_messages = [
    scam_message,
    legit_message,
    borderline_message,
    "Congratulations! You have been selected to receive a free NFT, claim your prize now before it expires!",
    "Anyone else having trouble syncing their hardware wallet after the latest firmware update?",
]

response = runtime.invoke_endpoint(
    EndpointName=ENDPOINT_NAME,
    ContentType="application/json",
    Body=json.dumps({"instances": [{"text": t} for t in batch_messages]}),
)

results = json.loads(response["Body"].read())

print(f"{'Message (truncated)':<70} {'Label':<12} Probability")
for msg, res in zip(batch_messages, results):
    truncated = (msg[:65] + "...") if len(msg) > 65 else msg
    print(f"{truncated:<70} {res['label']:<12} {res['probability']:.1%}")


Message (truncated)                                                    Label        Probability
URGENT: Your wallet has been selected for a guaranteed 100% profi...   Scam         99.9%
Been dollar-cost averaging into ETH for about a year now, curious...   Legitimate   10.5%
Hey, our community wallet is doing a small giveaway this week, ch...   Legitimate   20.9%
Congratulations! You have been selected to receive a free NFT, cl...   Scam         85.8%
Anyone else having trouble syncing their hardware wallet after th...   Legitimate   5.0%


## 7. Deploy a Baseline Endpoint for Demo Comparison

```text
Same SageMaker Pipeline (PreprocessData -> TrainModel -> AUCQualityGate -> RegisterModel)
        |
        +-- run with RegularizationC=10.0 (Section 2/3 above) --> champion endpoint (Section 5)
        |       iti113-team03-crypto-scam-detector
        |
        +-- run with RegularizationC=1.0  (this section)       --> baseline endpoint
                iti113-team03-crypto-scam-detector-baseline
```


### 7.0 Configuration

Reuses `pipeline`, `role`, `region` and `MODEL_PACKAGE_GROUP` already defined above — this section only needs a second endpoint name.


In [58]:
CHAMPION_ENDPOINT_NAME = ENDPOINT_NAME                    # existing, untouched by this section
BASELINE_ENDPOINT_NAME = f"{ENDPOINT_NAME}-baseline"      # new, created below

print(f"Champion endpoint (existing): {CHAMPION_ENDPOINT_NAME}")
print(f"Baseline endpoint (this section creates): {BASELINE_ENDPOINT_NAME}")


Champion endpoint (existing): iti113-team03-crypto-scam-detector
Baseline endpoint (this section creates): iti113-team03-crypto-scam-detector-baseline


### 7.1 Start a pipeline execution with baseline hyperparameters

`RegularizationC=1.0` with the defaults for everything else (`penalty=l2`, `solver=lbfgs`, `max_iter=1000`) is exactly the "Baseline Models" configuration from the Progress Check report's Section 5 — the same pipeline, same preprocessing, same quality gate as the champion, only the regularisation strength differs.


In [59]:
baseline_execution = pipeline.start(
    execution_display_name="baseline-c1-run",
    execution_description="Baseline: untuned logistic regression, C=1.0",
    parameters={
        'ModelType': 'logistic_regression',
        'RegularizationC': 1.0,
        'Penalty': 'l2',
        'Solver': 'lbfgs',
        'MaxIter': 1000,
        'QualityGateAUC': QUALITY_GATE_AUC,
    },
)
print(f'Baseline execution ARN: {baseline_execution.arn}')

Baseline execution ARN: arn:aws:sagemaker:ap-southeast-1:044528205969:pipeline/iti113-team03-crypto-scam-detector/execution/rhd54tp377a9


### 7.2 Wait for the pipeline execution to finish

In [60]:
prev = {}

while True:
    desc = baseline_execution.describe()
    status = desc["PipelineExecutionStatus"]

    steps = baseline_execution.list_steps()
    if isinstance(steps, dict):
        steps = steps.get("PipelineExecutionSteps", [])

    for step in steps:
        n = step["StepName"]
        s = step["StepStatus"]
        if prev.get(n) != s:
            print(f"{n:<20} {s}")
            prev[n] = s

    if status in ("Succeeded", "Failed", "Stopped"):
        print(f"\nPipeline Status: {status}")
        break

    time.sleep(30)


PreprocessData       Executing
TrainModel           Executing
PreprocessData       Succeeded
RegisterModel-RepackModel-0 Executing
AUCQualityGate       Succeeded
TrainModel           Succeeded
RegisterModel-RegisterModel Succeeded
RegisterModel-RepackModel-0 Succeeded

Pipeline Status: Succeeded


### 7.3 Approve the baseline model package

Same manual approval gate as the champion (Section 5) — this run's AUC-ROC (~0.89 per the Progress Check report) clears the 0.85 quality gate comfortably, so it registers automatically; approval is still required before deployment.


In [61]:
pkgs = sm.list_model_packages(
    ModelPackageGroupName=MODEL_PACKAGE_GROUP,
    SortBy='CreationTime', SortOrder='Descending', MaxResults=1
)['ModelPackageSummaryList']

if not pkgs:
    print('No model packages found. Check the pipeline execution above completed the Register step.')
else:
    baseline_pkg_arn = pkgs[0]['ModelPackageArn']
    print(f'Model package: {baseline_pkg_arn}\nStatus: {pkgs[0]["ModelApprovalStatus"]}')


Model package: arn:aws:sagemaker:ap-southeast-1:044528205969:model-package/team03-CryptoScamDetector/46
Status: PendingManualApproval


In [62]:
sm.update_model_package(ModelPackageArn=baseline_pkg_arn, ModelApprovalStatus='Approved')
print(f'Approved: {baseline_pkg_arn}')


Approved: arn:aws:sagemaker:ap-southeast-1:044528205969:model-package/team03-CryptoScamDetector/46


### 7.4 Deploy to a second serverless endpoint

Deploys to `BASELINE_ENDPOINT_NAME`, not `CHAMPION_ENDPOINT_NAME` — the champion endpoint and its approved package (Section 5) are untouched by this section.


In [63]:
try:
    sm.delete_endpoint(EndpointName=BASELINE_ENDPOINT_NAME)
    print(f'Deleted existing endpoint: {BASELINE_ENDPOINT_NAME}')
except sm.exceptions.ClientError:
    pass

try:
    sm.delete_endpoint_config(EndpointConfigName=BASELINE_ENDPOINT_NAME)
    print(f'Deleted existing endpoint config: {BASELINE_ENDPOINT_NAME}')
except sm.exceptions.ClientError:
    pass

baseline_deployable = ModelPackage(
    role=role, model_package_arn=baseline_pkg_arn, sagemaker_session=sagemaker.Session())

baseline_serverless_cfg = ServerlessInferenceConfig(memory_size_in_mb=2048, max_concurrency=5)

print(f'Deploying serverless endpoint: {BASELINE_ENDPOINT_NAME}')
baseline_predictor = baseline_deployable.deploy(
    serverless_inference_config=baseline_serverless_cfg,
    endpoint_name=BASELINE_ENDPOINT_NAME
)
print(f'Endpoint ready: {BASELINE_ENDPOINT_NAME}')


Deleted existing endpoint: iti113-team03-crypto-scam-detector-baseline
Deleted existing endpoint config: iti113-team03-crypto-scam-detector-baseline
Deploying serverless endpoint: iti113-team03-crypto-scam-detector-baseline
-----!Endpoint ready: iti113-team03-crypto-scam-detector-baseline


### 7.5 Compare champion vs baseline on the same messages

Sanity check before the demo: invokes both endpoints with the same messages and prints predictions side by side, confirming there's a visible improvement to show live.


In [64]:
print(f"{'Message':<12} {'Champion (C=10.0)':<28} {'Baseline (C=1.0)':<28}")
for label, text in [
    ("SCAM-STYLE", scam_message),
    ("LEGIT-STYLE", legit_message),
    ("BORDERLINE", borderline_message),
]:
    champ = invoke_scam_detector(text, endpoint_name=CHAMPION_ENDPOINT_NAME)
    base = invoke_scam_detector(text, endpoint_name=BASELINE_ENDPOINT_NAME)
    champ_str = f"{champ['label']} ({champ['probability']:.1%})"
    base_str = f"{base['label']} ({base['probability']:.1%})"
    print(f"{label:<12} {champ_str:<28} {base_str:<28}")


Message      Champion (C=10.0)            Baseline (C=1.0)            
SCAM-STYLE   Scam (99.9%)                 Scam (99.3%)                
LEGIT-STYLE  Legitimate (10.5%)           Legitimate (19.8%)          
BORDERLINE   Legitimate (20.9%)           Legitimate (23.9%)          


### 7.6 Wire into the relay and Streamlit

Notebook 06's relay exposes `POST /predict/baseline`, forwarding to `BASELINE_ENDPOINT_NAME`. Once this endpoint is deployed, restart Notebook 06's relay so it picks up the environment variable, and `pages/1_Scam_Detector.py`'s "Analyse Message" button will show champion and baseline predictions side by side automatically — no extra sidebar field needed, it derives the baseline route from the same relay URL.

### Cleanup — delete the baseline endpoint after your demo

Serverless endpoints don't charge for idle time, but delete this once you're done demoing so it isn't left running unnecessarily.


In [65]:
# Uncomment to delete the baseline endpoint after your demo/presentation.
# sm.delete_endpoint(EndpointName=BASELINE_ENDPOINT_NAME)
# sm.delete_endpoint_config(EndpointConfigName=BASELINE_ENDPOINT_NAME)
# print(f'Deleted baseline endpoint: {BASELINE_ENDPOINT_NAME}')


In [66]:
print(
    f'Pipeline: {PIPELINE_NAME}\n'
    f'MLflow: {MLFLOW_EXPERIMENT_NAME} on SageMaker MLflow App\n'
    f'MLflow App ARN: {MLFLOW_APP_ARN}\n'
    f'SageMaker Registry: {MODEL_PACKAGE_GROUP}\n'
    f'Champion endpoint: {CHAMPION_ENDPOINT_NAME} (Serverless)\n'
    f'Baseline endpoint: {BASELINE_ENDPOINT_NAME} (Serverless, Section 7)'
)


Pipeline: iti113-team03-crypto-scam-detector
MLflow: ITI113/team03/Experiment1 on SageMaker MLflow App
MLflow App ARN: arn:aws:sagemaker:ap-southeast-1:044528205969:mlflow-app/app-J5AYUG4AJHVW
SageMaker Registry: team03-CryptoScamDetector
Champion endpoint: iti113-team03-crypto-scam-detector (Serverless)
Baseline endpoint: iti113-team03-crypto-scam-detector-baseline (Serverless, Section 7)
